# My Data Science Project
---
Research Objectives

* To **analyze** the **temporal** and **spatial** **patterns** of **earthquake occurrences in the Philippines** **from 1900 to 2026** by examining their distribution over **time**, **geographic locations**, **depth characteristics**, and **earthquake-prone hotspots** using **geospatial analysis techniques**.
* To **investigate** the **relationships among earthquake characteristics and earthquake significance**, and to **identify the most important predictor variables influencing earthquake significance scores**.
* To **develop** and **evaluate** **machine learning models for predicting earthquake significance scores using earthquake-related features**, and to **assess model performance through Mean Absolute Error (MAE), Root Mean Square Error (RMSE), and Coefficient of Determination (R²)**

---

The notebook consist of the following:
* Data Loading, Data Cleaning, and Feature Extraction
* Temporal and Geospatial Earthquake Data Analysis
* Correlation Analysis of Features
* Hypothesis Testing
* Machine Learning Model Building, Training, and Evaluation




---



---
### Libraries and Machine Learning Tools


In [ ]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Data Preprocessing Techniques
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# For Splitting Dataset into Training Data and Test Data
from sklearn.model_selection import train_test_split

# For Model Building, Training, Evaluation, and Assessment of Models' Performance
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor

# Model Performance Indicator Tools
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

# Visualization Styles Configuration
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)


---
### Data Loading, Data Cleaning, and Feature Extraction

In [ ]:
df = pd.read_csv('Philippine_Earthquakes_USGS.csv')

df.info()

df = df.dropna(subset=['sig', 'mag', 'latitude', 'longitude'])

df['time'] = pd.to_datetime(df['time'], errors='coerce')

df['Year'] = df['time'].dt.year
df['Month'] = df['time'].dt.month
df['Quarter'] = df['time'].dt.quarter
df['DayOfWeek'] = df['time'].dt.dayofweek
df['Decade'] = (df['Year'] // 10) * 10

print(f"{df.shape[0]} records/rows and {df.shape[1]} features/columns.")
display(df[['time', 'mag', 'depth', 'latitude', 'longitude', 'sig']].head())

FileNotFoundError: [Errno 2] No such file or directory: 'Philippine_Earthquakes_USGS.csv'

---
### Temporal Analysis

In [ ]:
import matplotlib.ticker as ticker


plt.figure(figsize=(10, 6))
yearly_counts = df['Year'].value_counts().sort_index()
sns.lineplot(x=yearly_counts.index, y=yearly_counts.values, color='crimson')
plt.ylabel('Frequency')
plt.xlabel('Year')
plt.gca().xaxis.set_major_locator(ticker.MultipleLocator(10))
plt.tight_layout()
plt.show()


---
### Geospatial Analysis

In [ ]:
import geopandas as gpd
from matplotlib.lines import Line2D

plt.figure(figsize=(11, 13))
ax = plt.gca()

world_url = "https://gisco-services.ec.europa.eu/distribution/v2/countries/geojson/CNTR_RG_60M_2020_4326.geojson"
world = gpd.read_file(world_url)

world.plot(ax=ax, color='gainsboro', edgecolor='dimgray', alpha=0.8, zorder=1)

sns.scatterplot(
    x='longitude', y='latitude', data=df,
    color='gray', s=5, alpha=0.3, label='Recorded Earthquakes',
    ax=ax, zorder=2
)

sns.kdeplot(
    x=df['longitude'], y=df['latitude'],
    cmap='inferno', fill=True, bw_adjust=0.5, alpha=0.6, thresh=0.05, cbar_kws={'label': 'Seismic Event Density (Low to High)'},
    ax=ax, zorder=3
)

cities = {
    'Metro Manila': (120.98, 14.60),
    'Cebu City': (123.90, 10.31),
    'Davao City': (125.45, 7.19)
}

for city, (lon, lat) in cities.items():
    ax.plot(lon, lat, marker='*', color='cyan', markersize=14, markeredgecolor='black', zorder=5)
    ax.text(lon + 0.3, lat, city, fontsize=10, weight='bold', color='black',
            bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', pad=1.5), zorder=5)


ax.text(126.7, 9.5, 'PHILIPPINE TRENCH', color='darkred', weight='bold', fontsize=10, rotation=-75, alpha=0.6, zorder=4)
ax.text(119.0, 15.5, 'MANILA TRENCH', color='darkred', weight='bold', fontsize=10, rotation=75, alpha=0.6, zorder=4)

ax.set_xlim(115, 130)
ax.set_ylim(2, 22)

plt.xlabel('Longitude', fontsize=12)
plt.ylabel('Latitude', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.4, zorder=0)

handles, labels = ax.get_legend_handles_labels()
city_marker = Line2D([0], [0], marker='*', color='w', markerfacecolor='cyan', markeredgecolor='black', markersize=14, label='Major Cities')
handles.append(city_marker)
plt.legend(handles=handles, loc='upper left', framealpha=0.9, edgecolor='black')

plt.tight_layout()
plt.show()



---
### Correlation Analysis

In [ ]:
cols_to_correlate = ['sig', 'mag', 'depth', 'gap', 'rms', 'nst', 'dmin']
available_cols = [c for c in cols_to_correlate if c in df.columns]
corr_df = df[available_cols].dropna()

plt.figure(figsize=(10, 8))
corr_matrix = corr_df.corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.show()

---
### Hypothesis Testing

In [ ]:
print("--- Hypothesis Testing (Target Variable: SIG) ---")
for feature in ['mag', 'depth', 'gap', 'rms', 'nst', 'dmin']:
    if feature in corr_df.columns:
        stat, p_value = pearsonr(corr_df[feature], corr_df['sig'])
        print(f"Correlation between SIG and {feature.upper()}: r = {stat:.4f}, p-value = {p_value:.4e}")
        if p_value < 0.05:
            print(f"Result: Reject Null Hypothesis (H0) for {feature}. Significant relationship exists.\n")
        else:
            print(f"Result: Fail to reject Null Hypothesis (H0) for {feature}.\n")

---
### Machine Learning Model Building, Training, and Evaluation

In [ ]:
features = ['mag', 'depth', 'latitude', 'longitude', 'gap', 'rms', 'nst', 'dmin']
features = [f for f in features if f in df.columns]

X = df[features]
y = df['sig']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42, objective='reg:squarederror')
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('regressor', model)])

    pipeline.fit(X_train, y_train)
    trained_models[name] = pipeline

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R²': r2})

results_df = pd.DataFrame(results).sort_values(by='R²', ascending=False)
display(results_df)